In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import category_encoders as ce
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [34]:
df = pd.read_csv('train_test_network.csv')

In [35]:
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  object 
 1   src_port                211043 non-null  int64  
 2   dst_ip                  211043 non-null  object 
 3   dst_port                211043 non-null  int64  
 4   proto                   211043 non-null  object 
 5   service                 211043 non-null  object 
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  object 
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_ip_bytes        

In [37]:
df['type'].value_counts()

type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
scanning      20000
ransomware    20000
xss           20000
mitm           1043
Name: count, dtype: int64

# Preprocessing

### Missing Value

In [38]:
df.isna().sum().sum()

np.int64(0)

### Drop IP

In [39]:
df.filter(regex='(?i)ip')

,src_ip,dst_ip,src_ip_bytes,dst_ip_bytes,ssl_cipher
0,192.168.1.37,192.168.1.193,108064,3832,-
1,192.168.1.193,192.168.1.37,52,40,-
2,192.168.1.193,192.168.1.37,52,40,-
3,192.168.1.193,192.168.1.37,48,40,-
4,192.168.1.193,192.168.1.37,52,40,-
...,...,...,...,...,...
211038,192.168.1.32,176.28.50.165,2925,590,-
211039,192.168.1.32,176.28.50.165,2307,590,-
211040,192.168.1.32,176.28.50.165,4294,642,-
211041,192.168.1.32,176.28.50.165,2721,590,-


In [40]:
df_dropIP = df.drop(columns=['src_ip', 'dst_ip'])

In [41]:
df_dropIP.shape

(211043, 42)

### Feature Contain >90% Dash

In [42]:
threshold = 0.9

dash_ratio = (df_dropIP == '-').mean()
cols_under_90 = dash_ratio[(dash_ratio <= threshold) & (dash_ratio > 0)]
cols_over_90 = dash_ratio[dash_ratio > threshold]

print(f"Columns with '-' ratio under or equal to {threshold*100}%:")
print(cols_under_90)
print(f"\nColumns with '-' ratio over {threshold*100}%:")
print(cols_over_90)

Columns with '-' ratio under or equal to 90.0%:
service         0.625617
dns_query       0.834891
dns_AA          0.834095
dns_RD          0.834095
dns_RA          0.834095
dns_rejected    0.834095
dtype: float64

Columns with '-' ratio over 90.0%:
ssl_version             0.998100
ssl_cipher              0.998100
ssl_resumed             0.998100
ssl_established         0.998100
ssl_subject             0.999948
ssl_issuer              0.999948
http_trans_depth        0.998564
http_method             0.998640
http_uri                0.998640
http_version            0.998588
http_user_agent         0.998640
http_orig_mime_types    0.999924
http_resp_mime_types    0.999033
weird_name              0.998313
weird_addl              0.999256
weird_notice            0.998313
dtype: float64


In [43]:
df_drop_dash = df_dropIP.drop(columns=cols_over_90.index)

In [44]:
df_drop_dash = df_drop_dash.replace('-', np.nan)

In [45]:
df_drop_dash.shape

(211043, 26)

### Encoding

In [46]:
df_encoded = df_drop_dash.copy()

In [47]:
df_drop_dash["dns_query"].value_counts(normalize=True)

dns_query
a2z3kk2ebqzso7.iot.ap-southeast-2.amazonaws.com                             0.324810
testphp.vulnweb.com                                                         0.053035
elasticsearch                                                               0.050136
elasticsearch.mydns.com                                                     0.048386
_sleep-proxy._udp.local                                                     0.045688
                                                                              ...   
111.35.168.192.in-addr.arpa                                                 0.000029
a.b.a.c.3.6.f.f.1.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.2.0.f.f.ip6.arpa    0.000029
_nmea-0183._tcp.local                                                       0.000029
status.ws                                                                   0.000029
http://testphp.vulnweb.com/listproducts.php.hub                             0.000029
Name: proportion, Length: 725, dtype: float64

In [48]:
numeric_cols = df_drop_dash.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df_drop_dash.select_dtypes(include=['object']).columns

In [49]:
categorical_cols_wo_dns_query = categorical_cols.drop('dns_query')

In [50]:
for col in categorical_cols_wo_dns_query:
    print(f"Unique values in column '{col}':")
    print(df_drop_dash[col].unique())
    print("-" * 50)

Unique values in column 'proto':
['tcp' 'udp' 'icmp']
--------------------------------------------------
Unique values in column 'service':
[nan 'smb;gssapi' 'dce_rpc' 'smb' 'dns' 'ssl' 'http' 'ftp' 'gssapi']
--------------------------------------------------
Unique values in column 'conn_state':
['OTH' 'REJ' 'S1' 'RSTR' 'SF' 'RSTO' 'SH' 'S3' 'S0' 'SHR' 'S2' 'RSTOS0'
 'RSTRH']
--------------------------------------------------
Unique values in column 'dns_AA':
[nan 'F' 'T']
--------------------------------------------------
Unique values in column 'dns_RD':
[nan 'T' 'F']
--------------------------------------------------
Unique values in column 'dns_RA':
[nan 'T' 'F']
--------------------------------------------------
Unique values in column 'dns_rejected':
[nan 'F' 'T']
--------------------------------------------------
Unique values in column 'type':
['backdoor' 'ddos' 'dos' 'injection' 'mitm' 'normal' 'password'
 'ransomware' 'scanning' 'xss']
---------------------------------------

#### dns_query

In [51]:
df_encoded['dns_query'].isna().sum()

np.int64(176198)

In [52]:
df_encoded["has_dns"] = df_encoded["dns_query"].notna().astype(int)
df_encoded = df_encoded.drop(columns=['dns_query'])
df_encoded['has_dns'].value_counts()

has_dns
0    176198
1     34845
Name: count, dtype: int64

#### Binary Cols

In [53]:
binary_cols = ['dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected']

for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({'T': 1, 'F': 0})

#### One Hot Encoding

In [54]:
df_encoded = pd.get_dummies(
    df_encoded,
    columns=['proto'],
    drop_first=False  # kalau pakai tree model
)

#### Target Feature

In [55]:
le = LabelEncoder()
df_encoded['type'] = le.fit_transform(df_encoded['type'])

#### Target Encoding

In [56]:
target_cols = ['service', 'conn_state']

encoder = ce.TargetEncoder(cols=target_cols)

df_encoded[target_cols] = encoder.fit_transform(
    df_encoded[target_cols],
    df_encoded['type']
)

### Split

In [57]:
x_train, x_test, y_train, y_test = train_test_split(
    df_encoded.drop(columns=['type']),
    df_encoded['type'],
    test_size=0.2,
    random_state=42,
    stratify=df_encoded['type']
)

# Modelling

In [58]:
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

In [59]:
xgb_model.fit(x_train, y_train)

c:\Users\PREDATOR\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:199: UserWarning: [13:07:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [60]:
y_pred = xgb_model.predict(x_test)

In [61]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9900495155061717
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4000
           1       0.99      0.98      0.99      4000
           2       0.99      0.98      0.98      4000
           3       0.98      0.97      0.98      4000
           4       0.72      0.82      0.77       209
           5       1.00      1.00      1.00     10000
           6       0.99      0.99      0.99      4000
           7       1.00      1.00      1.00      4000
           8       0.99      0.99      0.99      4000
           9       0.97      0.99      0.98      4000

    accuracy                           0.99     42209
   macro avg       0.96      0.97      0.97     42209
weighted avg       0.99      0.99      0.99     42209



# Save Model

In [ ]:
# save model
joblib.dump(xgb_model, "models/model.pkl")

# save encoders
joblib.dump(encoder, "models/target_encoder.pkl")
joblib.dump(le, "models/label_encoder.pkl")

# save feature columns
joblib.dump(x_train.columns.tolist(), "models/feature_columns.pkl")

['model/feature_columns.pkl']